# Homework 3 - Machine Learning for Classification



In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation



In [2]:
df_full = df.copy()
df_full['converted'].value_counts(normalize=True)

converted
1    0.5766
0    0.4234
Name: proportion, dtype: float64

In [3]:
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = df.drop(columns=['converted']).select_dtypes(include='number').columns.tolist()

df_clean = df.copy()
df_clean[cat_cols] = df_clean[cat_cols].fillna('NA')
df_clean[num_cols] = df_clean[num_cols].fillna(0.0)

print('Categoricas:', cat_cols)
print('Numericas:', num_cols)

Categoricas: ['lead_source', 'industry', 'employment_status', 'location']
Numericas: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


## Q1. Mode of `industry`



In [4]:
df['industry'].mode()[0]

'technology'

## Q2. Correlation matrix (numerical features)


In [5]:
corr = df_clean[num_cols].corr().abs()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [6]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for a, b in pairs:
    print(f"{a:>28} y {b:<28} -> {corr.loc[a, b]:.4f}")

best = max(pairs, key=lambda p: corr.loc[p[0], p[1]])
print('\n-> Mayor correlacion:', best)

           interaction_count y lead_score                   -> 0.9157
    number_of_courses_viewed y lead_score                   -> 0.7572
    number_of_courses_viewed y interaction_count            -> 0.7216
               annual_income y interaction_count            -> 0.1228

-> Mayor correlacion: ('interaction_count', 'lead_score')


## Split the data



In [7]:
y = df['converted']
X = df_clean.drop(columns=['converted'])

X_full_train, X_test = train_test_split(df_clean, test_size=0.2, random_state=42)
X_train, X_val = train_test_split(X_full_train, test_size=0.25, random_state=42)

y_train = X_train.pop('converted')
y_val = X_val.pop('converted')
y_test = X_test.pop('converted')

print(len(X_train), len(X_val), len(X_test))

3000 1000 1000


## Helper: one-hot encoding + logistic regression

In [8]:
def train_model(X_tr, y_tr, X_va, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr_vec = dv.fit_transform(X_tr.to_dict(orient='records'))
    X_va_vec = dv.transform(X_va.to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_tr_vec, y_tr)

    return dv, model, X_va_vec

## Q3. Mutual information (`converted` vs categorical)


In [9]:
for col in cat_cols:
    score = mutual_info_score(y_train, X_train[col])
    print(f'{col:>20}: {score:.2f}')

         lead_source: 0.03
            industry: 0.00
   employment_status: 0.02
            location: 0.00


## Q4. Logistic regression accuracy


In [10]:
dv, model, X_val_vec = train_model(X_train, y_train, X_val)
y_pred = model.predict(X_val_vec)
accuracy = accuracy_score(y_val, y_pred)
print('-> Accuracy:', round(accuracy, 2))

-> Accuracy: 0.65


## Q5. Feature elimination


In [11]:
for feat in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    X_tr = X_train.drop(columns=[feat])
    X_va = X_val.drop(columns=[feat])
    _, m, vec = train_model(X_tr, y_train, X_va)
    acc = accuracy_score(y_val, m.predict(vec))
    print(f'{feat:>25}: acc={acc:.4f} diff={abs(accuracy - acc):.4f}')

print('\n-> Menor diferencia: number_of_courses_viewed')

              lead_source: acc=0.6420 diff=0.0030


 number_of_courses_viewed: acc=0.6430 diff=0.0020


        interaction_count: acc=0.6010 diff=0.0440

-> Menor diferencia: number_of_courses_viewed


## Q6. Regularized logistic regression


In [12]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    _, m, vec = train_model(X_train, y_train, X_val, C=C)
    acc = accuracy_score(y_val, m.predict(vec))
    print(f'C = {C:<8} acc = {acc:.3f}')
    
print('\n-> Mejor C: 0.001')

C = 1e-06    acc = 0.598


C = 1e-05    acc = 0.598


C = 0.0001   acc = 0.613


C = 0.001    acc = 0.645

-> Mejor C: 0.001
